### LT_MED in Pyomo
Converted from PSA's LT-MED general design model  
Created on June, 2022  
Author: Zhuoran Zhang  


In [1]:
from pyomo.environ import ConcreteModel, Objective, value, Var, Param, Constraint, Set,Expression, Objective, exp,\
        check_optimal_termination, SolverFactory, units as pyunits

from pyomo.util.check_units import assert_units_consistent
from idaes.core.util.model_statistics import degrees_of_freedom

import numpy as np
import pandas as pd

from SW_functions import SW_Density, BPE, SW_Enthalpy
import IAPWS97_thermo_functions as TD_func

###  **Empirical model**  


In [2]:
'''
Instantiate model for LT-MED empirical model
'''
m = ConcreteModel()


'''
Add Vars and Param for 6 model inputs
'''
m.Xf = Var(initialize = 40000,
                     bounds = (30000, 60000),
                     units = pyunits.mg / pyunits.L,
                     doc = 'Feedwater salinity (g/L)')

m.Ts = Var(initialize = 80,
                     bounds = (60, 85),
                     units = pyunits.C,
                     doc = 'The temperature of the steam at the inlet of the first bundle tube (C)')

# Interger should be picked from [3,6,9,12,14] for Nef
# Param object is selected for modeling purpose 

m.Nef = Param(initialize = 12,
              doc = 'Number of effects')



m.Capacity = Var(initialize = 2000,
                     bounds = (2000, None),
                     units = pyunits.m**3 / pyunits.d,
                     doc = 'Capacity of the plant (m3/day)')

m.Tin = Var(initialize = 25,
                     bounds = (15, 35),
                     units = pyunits.C,
                     doc = 'Condenser inlet seawater temperature (C)')

m.RR = Var(initialize = 0.50,
                     bounds = (0.30, 0.50),
                     doc = 'Recovery ratio')


'''
Add Vars for model outputs
'''

m.P_req = Var(initialize = 5000,
                     bounds = (0, None),
                     units = pyunits.kJ / pyunits.s,
                     doc = 'Thermal power requirement (kW)')

m.sA = Var(initialize = 2,
           bounds = (0, None),
           units = pyunits.m**2 / (pyunits.m**3 / pyunits.d),
           doc = 'Specific area (m2/m3/day))')

m.STEC = Var(initialize = 65,
             bounds = (0, None),
             units = pyunits.kWh / pyunits.m**3,
             doc = 'Specific thermal power consumption (kWh/m3)')
                    

m.GOR = Var(initialize = 10,
            bounds = (0, None),
            doc = 'Gained output ratio')

m.qF = Var(initialize = 200,
           bounds = (0, None),
           units = pyunits.m**3 / pyunits.h,
           doc = 'Feed water flow rate (m3/h)')
                    
m.qs = Var(initialize = 2,
           bounds = (0, None),
           units = pyunits.kg / pyunits.h,
           doc = 'Steam flow rate (kg/h)')
                    
'''
Add Vars for intermediate model variables
'''
m.TN = Var(initialize = 35, 
           bounds = (25, 45),
           units = pyunits.C,
           doc = 'Last effect vapor temperature')

m.T_d = Var(initialize = 35, 
           bounds = (25, 45),
           units = pyunits.C,
           doc = 'Distillate temperature')

m.q_d = Var(initialize = 20, 
           bounds = (0, None),
           units = pyunits.m**3 / pyunits.h,
           doc = 'Distillate flow rate (m3/hr)')

m.q_b = Var(initialize = 20, 
           bounds = (0, None),
           units = pyunits.m**3 / pyunits.h,
           doc = 'Brine flow rate (m3/hr)')


m.s_b = Var(initialize = 60, 
           bounds = (0, None),
           units = pyunits.g / pyunits.L,
           doc = 'Brine salinity (g/L)')

m.SW_BPE = Var(initialize = 0.5, 
           bounds = (0, None),
           units = pyunits.C,
           doc = 'Boiling point elevation (C)')


m.T_b = Var(initialize = 35, 
           bounds = (0, None),
           units = pyunits.C,
           doc = 'Brine temperature')

m.T_cool = Var(initialize = 42, 
           bounds = (0, None),
           units = pyunits.C,
           doc = 'Reject cooling water temperature')

m.rho_b = Var(initialize = 1000,
              bounds = (0, None),
              units = pyunits.kg / pyunits.m**3,
              doc = 'Brine density (kg/m3)')

m.rho_d = Var(initialize = 1000,
              bounds = (0, None),
              units = pyunits.kg / pyunits.m**3,
              doc = 'Distillate density (kg/m3)')

m.rho_sw = Var(initialize = 1000,
              bounds = (0, None),
              units = pyunits.kg / pyunits.m**3,
              doc = 'Seawater density (kg/m3)')

m.rho_f = Var(initialize = 1000,
              bounds = (0, None),
              units = pyunits.kg / pyunits.m**3,
              doc = 'Reject cooling water density (kg/m3)')

m.h_b = Var(initialize = 1000,
            bounds = (0, None),
            units = pyunits.kJ / pyunits.kg,
            doc = 'Brine enthalpy (kJ/kg)')

m.h_d = Var(initialize = 1000,
            bounds = (0, None),
            units = pyunits.kJ / pyunits.kg,
            doc = 'Distillate enthalpy (kJ/kg)')

m.h_sw = Var(initialize = 1000,
            bounds = (0, None),
            units = pyunits.kJ / pyunits.kg,
            doc = 'Seawater enthalpy (kJ/kg)')

m.h_cool = Var(initialize = 800,
              bounds = (0, None),
              units = pyunits.kJ / pyunits.kg,
              doc = 'Reject cooling water enthalpy (kJ/kg)')

m.m_b = Var(initialize = 1000,
              bounds = (0, None),
              units = pyunits.kg / pyunits.s,
              doc = 'Brine mass flow rate (kg/s)')

m.m_d = Var(initialize = 1000,
              bounds = (0, None),
              units = pyunits.kg / pyunits.s,
              doc = 'Distillate mass flow rate (kg/s)')

m.m_f = Var(initialize = 1000,
              bounds = (0, None),
              units = pyunits.kg / pyunits.s,
              doc = 'Feed water mass flow rate (kg/s)')

m.m_sw = Var(initialize = 1000,
              bounds = (0, None),
              units = pyunits.kg / pyunits.s,
              doc = 'Intake water mass flow rate (kg/s)')

m.q_sw = Var(initialize = 1000,
              bounds = (0, None),
              units = pyunits.m **3 / pyunits.h,
              doc = 'Intake water volume flow rate (m3/h)')

m.q_cooling = Var(initialize = 1000,
              bounds = (None, None),
              units = pyunits.m **3 / pyunits.h,
              doc = 'Cooling water volume flow rate (m3/h)')


'''
Add Params for parameters
'''
# Coefficients for GOR calculation
Nef_vals = [3, 6, 9, 12, 14]  # Number of effects
coeffs_set = range(15)        # Number of coefficients

GOR_coeffs = {3:[1.60E-07,0.826895712,-2.04E-07,0.003340838,-5.56E-09,0.000666667,-0.003295958,1.17E-10,-0.000549708,-2.46E-06,2.662545127,-1.98E-07,7.41E-07,-0.675925926,-4.12E-13],
              6:[5.86E-07,2.940942982,-1.44E-06,0.007985234,-2.26E-08,0.001472222,-0.007157144,8.73E-09,-0.001540936,4.88E-06,4.741753363,-1.04E-06,-1.67E-05,-2.333333333,-7.41E-13],
              9:[1.67E-06,5.9507846,-3.94E-06,0.012607651,-5.50E-08,0.002222222,-0.010203548,2.47E-08,-0.002549708,2.94E-05,6.350104873,-1.05E-05,-5.09E-05,-4.653703704,-2.88E-12],
              12:[3.30E-06,9.621851852,-7.98E-06,0.016637037,-1.09E-07,0.002,-0.012637326,5.13E-08,-0.003277778,8.28E-05,7.592772368,-3.09E-05,-9.98E-05,-7.425925926,-6.09E-12],
              14:[5.27E-06,12.44928443,-1.15E-05,0.019398098,-1.59E-07,0.001666667,-0.013396636,7.88E-08,-0.003333333,0.000121663,8.195669495,-5.34E-05,-0.00013251,-9.627577763,-1.80E-11]
             }

def GOR_coeffs_init(m, Nef_vals, coeffs_set):
    return GOR_coeffs[Nef_vals][coeffs_set]

m.GOR_coeffs = Param(Nef_vals, coeffs_set, initialize = GOR_coeffs_init)

# Coefficients for sA calculation
Nef_vals1 = [3, 6, 9]
Nef_vals2 = [12, 14]

coeffs_set1 = range(35)        # Number of coefficients for N=3, 6, 9
coeffs_set2 = range(70)        # Number of coefficients for N=12, 14

sA_coeffs = {3:[0.000596217,-3.66E-09,0,-2.44E-05,1.93E-09,0,5.60E-05,0,-2.95E-07,5.30E-11,0,7.14E-06,0,0.064807392,2.06E-07,0.00974051,0,-1.16E-05,-3.96E-11,0,-5.27E-06,0,-0.05718687,-2.61E-07,-0.011936049,-0.000702529,0.013464849,1.65E-07,0.003686623,0.000759933,0,-0.00019293,-0.000182949,0,3.20E-14],
             6:[0.00040105,-6.57E-09,0,-1.56E-05,3.67E-10,0,2.62E-05,0,7.08E-07,8.73E-12,0,1.46E-06,0,0.032775092,5.04E-08,0.002499309,0,-3.30E-06,-6.53E-12,0,-1.02E-06,0,-0.028641745,-6.66E-08,-0.002735652,-0.000301667,0.005600544,4.10E-08,0.000713386,0.000329733,0,-7.31E-05,-0.00010089,0,4.94E-14],
             9:[0.000596217,-3.66E-09,0,-2.44E-05,1.93E-09,0,5.60E-05,0,-2.95E-07,5.30E-11,0,7.14E-06,0,0.064807392,2.06E-07,0.00974051,0,-1.16E-05,-3.96E-11,0,-5.27E-06,0,-0.05718687,-2.61E-07,-0.011936049,-0.000702529,0.013464849,1.65E-07,0.003686623,0.000759933,0,-0.00019293,-0.000182949,0,3.20E-14],
             12:[0.000000E+00,3.304374E-08,-6.761157E-13,0.000000E+00,0.000000E+00,-5.496094E-09,1.958695E-13,0.000000E+00,0.000000E+00,6.105760E-09,0.000000E+00,0.000000E+00,0.000000E+00,0.000000E+00,-8.520444E-10,2.227182E-14,0.000000E+00,0.000000E+00,6.610470E-10,0.000000E+00,0.000000E+00,0.000000E+00,0.000000E+00,3.561099E-06,9.693068E-12,0.000000E+00,1.148815E-06,0.000000E+00,0.000000E+00,-1.049434E-08,0.000000E+00,0.000000E+00,0.000000E+00,1.734819E-10,-1.367980E-14,0.000000E+00,0.000000E+00,-5.044097E-10,0.000000E+00,0.000000E+00,0.000000E+00,0.000000E+00,-2.717657E-06,-3.905605E-11,0.000000E+00,-1.397796E-06,0.000000E+00,0.000000E+00,-4.132341E-08,0.000000E+00,0.000000E+00,0.000000E+00,4.380618E-07,2.072263E-11,0.000000E+00,4.398758E-07,0.000000E+00,0.000000E+00,5.991695E-08,0.000000E+00,0.000000E+00,0.000000E+00,-1.833849E-08,0.000000E+00,-3.505028E-06,0.000000E+00,1.713226E-06,0.000000E+00,0.000000E+00,6.273434E-18],
             14:[0.000000E+00,4.368251E-08,2.260942E-13,0.000000E+00,0.000000E+00,-4.762111E-08,1.282504E-12,0.000000E+00,0.000000E+00,3.611544E-08,0.000000E+00,0.000000E+00,0.000000E+00,0.000000E+00,-3.197411E-09,8.656950E-14,0.000000E+00,0.000000E+00,3.617982E-09,0.000000E+00,0.000000E+00,0.000000E+00,0.000000E+00,6.034818E-06,4.276140E-11,0.000000E+00,4.908627E-06,0.000000E+00,0.000000E+00,-1.357859E-08,0.000000E+00,0.000000E+00,0.000000E+00,1.131144E-10,-5.279738E-14,0.000000E+00,0.000000E+00,-2.836528E-09,0.000000E+00,0.000000E+00,0.000000E+00,0.000000E+00,-3.906973E-06,-1.642642E-10,0.000000E+00,-6.562415E-06,0.000000E+00,0.000000E+00,-1.071227E-07,0.000000E+00,0.000000E+00,0.000000E+00,7.876487E-07,8.941839E-11,0.000000E+00,2.276216E-06,0.000000E+00,0.000000E+00,1.772933E-07,0.000000E+00,0.000000E+00,0.000000E+00,-6.455677E-08,0.000000E+00,-1.423548E-05,0.000000E+00,7.012498E-06,0.000000E+00,0.000000E+00,1.716278E-19]
             }
            
def sA_coeffs1_init(m, Nef_vals1, coeffs_set1):
    return sA_coeffs[Nef_vals1][coeffs_set1]
def sA_coeffs2_init(m, Nef_vals2, coeffs_set2):
    return sA_coeffs[Nef_vals2][coeffs_set2]

# Initialization 
m.sA_coeffs1 = Param(Nef_vals1, coeffs_set1, initialize = sA_coeffs1_init)
m.sA_coeffs2 = Param(Nef_vals2, coeffs_set2, initialize = sA_coeffs2_init)


m.Q_loss = Param(initialize = 0.054,
                 doc = ' System thermal loss')


In [3]:
'''
Add Constraints 
'''
# Assumption: Last effect vapor temperature is 10 degree higher than condenser inlet temperature
m.TN_Tin = Constraint(expr = m.TN == m.Tin + 10)
# Assumption: Distillate temperature is the same as last effect vapor temperature
m.Td_TN = Constraint(expr = m.T_d == m.TN)

# GOR calculation
m.GOR_cal = Constraint(expr = \
           m.GOR == m.Xf * m.GOR_coeffs[(m.Nef, 0)]      + m.RR * m.GOR_coeffs[(m.Nef, 1)]      + m.Xf*m.RR * m.GOR_coeffs[(m.Nef, 2)] \
                  + m.TN * m.GOR_coeffs[(m.Nef, 3)]      + m.TN*m.Xf * m.GOR_coeffs[(m.Nef, 4)] +  m.TN*m.RR * m.GOR_coeffs[(m.Nef, 5)] \
                  + m.Ts * m.GOR_coeffs[(m.Nef, 6)]      + m.Ts*m.Xf * m.GOR_coeffs[(m.Nef, 7)] +  m.Ts*m.RR * m.GOR_coeffs[(m.Nef, 8)] \
                  + m.Ts*m.TN * m.GOR_coeffs[(m.Nef, 9)] + 1 * m.GOR_coeffs[(m.Nef, 10)]        + m.Ts**2 * m.GOR_coeffs[(m.Nef, 11)] \
                  + m.TN**2 * m.GOR_coeffs[(m.Nef, 12)]  + m.RR**2 * m.GOR_coeffs[(m.Nef, 13)]  + m.Xf**2 * m.GOR_coeffs[(m.Nef, 14)] 
                    )

# sA calculation
if m.Nef in [3,6,9]:
    m.sA_cal1 = Constraint(expr = \
               m.sA == m.Xf * m.sA_coeffs1[(m.Nef, 0)]            + m.Xf**2 * m.sA_coeffs1[(m.Nef, 1)]       + m.RR * m.sA_coeffs1[(m.Nef, 2)] \
                    +  m.RR*m.Xf * m.sA_coeffs1[(m.Nef, 3)]       + m.RR*m.Xf**2 * m.sA_coeffs1[(m.Nef, 4)]  + m.RR**2 * m.sA_coeffs1[(m.Nef, 5)] \
                    +  m.RR**2*m.Xf * m.sA_coeffs1[(m.Nef, 6)]    + m.TN * m.sA_coeffs1[(m.Nef, 7)]          + m.TN*m.Xf * m.sA_coeffs1[(m.Nef, 8)] \
                    +  m.TN*m.Xf**2 * m.sA_coeffs1[(m.Nef, 9)]    + m.TN*m.RR * m.sA_coeffs1[(m.Nef, 10)]    + m.TN*m.RR*m.Xf * m.sA_coeffs1[(m.Nef, 11)] \
                    +  m.TN*m.RR**2 * m.sA_coeffs1[(m.Nef, 12)]   + m.TN**2 * m.sA_coeffs1[(m.Nef, 13)]      + m.TN**2*m.Xf* m.sA_coeffs1[(m.Nef, 14)] \
                    +  m.TN**2*m.RR * m.sA_coeffs1[(m.Nef, 15)]   + m.Ts * m.sA_coeffs1[(m.Nef, 16)]         + m.Ts*m.Xf * m.sA_coeffs1[(m.Nef, 17)] \
                    +  m.Ts*m.Xf**2 * m.sA_coeffs1[(m.Nef, 18)]   + m.Ts*m.RR * m.sA_coeffs1[(m.Nef, 19)]    + m.Ts*m.RR*m.Xf * m.sA_coeffs1[(m.Nef, 20)] \
                    +  m.Ts*m.RR**2 * m.sA_coeffs1[(m.Nef, 21)]   + m.Ts*m.TN * m.sA_coeffs1[(m.Nef, 22)]    + m.Ts*m.TN*m.Xf * m.sA_coeffs1[(m.Nef, 23)] \
                    +  m.Ts*m.TN*m.RR * m.sA_coeffs1[(m.Nef, 24)] + m.Ts*m.TN**2 * m.sA_coeffs1[(m.Nef, 25)] + m.Ts**2 * m.sA_coeffs1[(m.Nef, 26)] \
                    +  m.Ts**2*m.Xf * m.sA_coeffs1[(m.Nef, 27)]   + m.Ts**2*m.RR * m.sA_coeffs1[(m.Nef, 28)] + m.Ts**2*m.TN * m.sA_coeffs1[(m.Nef, 29)] \
                    +  1 * m.sA_coeffs1[(m.Nef, 30)]              + m.Ts**3* m.sA_coeffs1[(m.Nef, 31)]       + m.TN**3 * m.sA_coeffs1[(m.Nef, 32)] \
                    +  m.RR**3 * m.sA_coeffs1[(m.Nef, 33)]        + m.Xf**3 * m.sA_coeffs1[(m.Nef, 34)] 
                     )    
if m.Nef in [12, 14]:
    m.sA_cal2 = Constraint(expr = \
               m.sA == m.Xf * m.sA_coeffs2[(m.Nef, 0)]                 + m.Xf**2 * m.sA_coeffs2[(m.Nef, 1)]            + m.Xf**3 * m.sA_coeffs2[(m.Nef, 2)] \
                    +  m.RR * m.sA_coeffs2[(m.Nef, 3)]                 + m.RR*m.Xf * m.sA_coeffs2[(m.Nef, 4)]          + m.RR*m.Xf**2 * m.sA_coeffs2[(m.Nef, 5)] \
                    +  m.RR*m.Xf**3* m.sA_coeffs2[(m.Nef, 6)]          + m.RR**2 * m.sA_coeffs2[(m.Nef, 7)]            + m.RR**2*m.Xf * m.sA_coeffs2[(m.Nef, 8)] \
                    +  m.RR**2*m.Xf**2 * m.sA_coeffs2[(m.Nef, 9)]      + m.RR**3 * m.sA_coeffs2[(m.Nef, 10)]           + m.RR**3*m.Xf * m.sA_coeffs2[(m.Nef, 11)] \
                    +  m.TN * m.sA_coeffs2[(m.Nef, 12)]                + m.TN*m.Xf * m.sA_coeffs2[(m.Nef, 13)]         + m.TN*m.Xf**2 * m.sA_coeffs2[(m.Nef, 14)] \
                    +  m.TN*m.Xf**3 * m.sA_coeffs2[(m.Nef, 15)]        + m.TN*m.RR * m.sA_coeffs2[(m.Nef, 16)]         + m.TN*m.RR*m.Xf * m.sA_coeffs2[(m.Nef, 17)] \
                    +  m.TN*m.RR*m.Xf**2 * m.sA_coeffs2[(m.Nef, 18)]   + m.TN*m.RR**2 * m.sA_coeffs2[(m.Nef, 19)]      + m.TN*m.RR**2*m.Xf * m.sA_coeffs2[(m.Nef, 20)] \
                    +  m.TN*m.RR**3 * m.sA_coeffs2[(m.Nef, 21)]        + m.TN**2* m.sA_coeffs2[(m.Nef, 22)]            + m.TN**2*m.Xf * m.sA_coeffs2[(m.Nef, 23)] \
                    +  m.TN**2*m.Xf**2 * m.sA_coeffs2[(m.Nef, 24)]     + m.TN**2*m.RR * m.sA_coeffs2[(m.Nef, 25)]      + m.TN**2*m.RR*m.Xf * m.sA_coeffs2[(m.Nef, 26)] \
                    +  m.TN**2*m.RR**2 * m.sA_coeffs2[(m.Nef, 27)]     + m.TN**3 * m.sA_coeffs2[(m.Nef, 28)]           + m.TN**3*m.Xf* m.sA_coeffs2[(m.Nef, 29)] \
                    +  m.TN**3*m.RR * m.sA_coeffs2[(m.Nef, 30)]        + m.Ts * m.sA_coeffs2[(m.Nef, 31)]              + m.Ts*m.Xf * m.sA_coeffs2[(m.Nef, 32)] \
                    +  m.Ts*m.Xf**2 * m.sA_coeffs2[(m.Nef, 33)]        + m.Ts*m.Xf**3 * m.sA_coeffs2[(m.Nef, 34)]      + m.Ts*m.RR * m.sA_coeffs2[(m.Nef, 35)] \
                    +  m.Ts*m.RR*m.Xf * m.sA_coeffs2[(m.Nef, 36)]      + m.Ts*m.RR*m.Xf**2 * m.sA_coeffs2[(m.Nef, 37)] + m.Ts*m.RR**2 * m.sA_coeffs2[(m.Nef, 38)] \
                    +  m.Ts*m.RR**2*m.Xf * m.sA_coeffs2[(m.Nef, 39)]   + m.Ts*m.RR**3 * m.sA_coeffs2[(m.Nef, 40)]      + m.Ts*m.TN* m.sA_coeffs2[(m.Nef, 41)] \
                    +  m.Ts*m.TN*m.Xf * m.sA_coeffs2[(m.Nef, 42)]      + m.Ts*m.TN*m.Xf**2 * m.sA_coeffs2[(m.Nef, 43)] + m.Ts*m.TN*m.RR * m.sA_coeffs2[(m.Nef, 44)] \
                    +  m.Ts*m.TN*m.RR*m.Xf * m.sA_coeffs2[(m.Nef, 45)] + m.Ts*m.TN*m.RR**2 * m.sA_coeffs2[(m.Nef, 46)] + m.Ts*m.TN**2* m.sA_coeffs2[(m.Nef, 47)] \
                    +  m.Ts*m.TN**2*m.Xf * m.sA_coeffs2[(m.Nef, 48)]   + m.Ts*m.TN**2*m.RR * m.sA_coeffs2[(m.Nef, 49)] + m.Ts*m.TN**3 * m.sA_coeffs2[(m.Nef, 50)] \
                    +  m.Ts**2 * m.sA_coeffs2[(m.Nef, 51)]             + m.Ts**2*m.Xf * m.sA_coeffs2[(m.Nef, 52)]      + m.Ts**2*m.Xf**2 * m.sA_coeffs2[(m.Nef, 53)] \
                    +  m.Ts**2*m.RR * m.sA_coeffs2[(m.Nef, 54)]        + m.Ts**2*m.RR*m.Xf * m.sA_coeffs2[(m.Nef, 55)] + m.Ts**2*m.RR**2 * m.sA_coeffs2[(m.Nef, 56)] \
                    +  m.Ts**2*m.TN * m.sA_coeffs2[(m.Nef, 57)]        + m.Ts**2*m.TN*m.Xf * m.sA_coeffs2[(m.Nef, 58)] + m.Ts**2*m.TN*m.RR * m.sA_coeffs2[(m.Nef, 59)] \
                    +  m.Ts**2*m.TN**2 * m.sA_coeffs2[(m.Nef, 60)]     + m.Ts**3 * m.sA_coeffs2[(m.Nef, 61)]           + m.Ts**3*m.Xf * m.sA_coeffs2[(m.Nef, 62)] \
                    +  m.Ts**3*m.RR* m.sA_coeffs2[(m.Nef, 63)]         + m.Ts**3*m.TN * m.sA_coeffs2[(m.Nef, 64)]      + 1 * m.sA_coeffs2[(m.Nef, 65)] \
                    +  m.Ts**4 * m.sA_coeffs2[(m.Nef, 66)]             + m.TN**4 * m.sA_coeffs2[(m.Nef, 67)]           + m.RR**4 * m.sA_coeffs2[(m.Nef, 68)] \
                    +  m.Xf**4 * m.sA_coeffs2[(m.Nef, 69)] 
                     )

# Feed flow rate calculation
m.qF_cal = Constraint(expr = m.qF == m.Capacity / m.RR / 24)

# Steam flow rate calculation
m.qs_cal = Constraint(expr = m.qs == m.Capacity * 1000 / m.GOR / 24 / 3600)

# Mass balance
m.q_d_cal = Constraint(expr = m.q_d == m.Capacity / 24)
m.q_b_cal = Constraint(expr = m.q_b == m.qF - m.q_d)
m.s_b_cal = Constraint(expr = m.s_b == m.Xf / 1000 / (1 - m.RR))

# BPE calculation
m.SW_BPE_cal = Constraint(expr = m.SW_BPE == BPE(m.T_d, m.s_b))

# Brine temperature
m.T_b_cal = Constraint(expr = m.T_b == m.T_d + m.SW_BPE)

# Assumption: The temperature difference between the inlet and outlet seawater temperature in the condenser is 7 degC
m.T_cool_cal = Constraint(expr = m.T_cool == m.Tin + 7)

# Densities
m.rho_b_cal = Constraint(expr = m.rho_b == SW_Density(m.T_b,'c',m.s_b*1000,'ppm',1,'bar'))  # brine density
m.rho_d_cal = Constraint(expr = m.rho_d == SW_Density(m.T_d, 'c', 0, 'ppm', 1, 'bar')) # distillate density
m.rho_sw_cal= Constraint(expr = m.rho_sw == SW_Density(m.Tin,'c',m.Xf,'ppm',1,'bar'))  # seawater density
m.rho_f_cal = Constraint(expr = m.rho_f == SW_Density(m.T_cool,'c',m.Xf,'ppm',1,'bar'))  # cooling reject density

# enthalpies
m.h_d_cal = Constraint(expr = m.h_d == TD_func.enthalpySatLiqTW(m.T_d+273.15))
m.h_b_cal = Constraint(expr = m.h_b == SW_Enthalpy(m.T_b, m.s_b)/ 1000 )
m.h_sw_cal = Constraint(expr = m.h_sw == SW_Enthalpy(m.Tin, m.Xf/1000)/ 1000 )
m.h_cool_cal = Constraint(expr = m.h_cool == SW_Enthalpy(m.T_cool, m.Xf/1000)/ 1000 )

# Energy consumption
m.STEC_cal = Constraint(expr = m.STEC == 1 / m.GOR * (TD_func.enthalpySatVapTW(m.Ts + 273.15) - TD_func.enthalpySatLiqTW(m.Ts+273.15)) * m.rho_d /3600)
m.P_req_cal= Constraint(expr = m.P_req == m.STEC * m.Capacity / 24)
                         
# Mass flow rates
m.m_d_cal = Constraint(expr = m.m_d == m.q_d * m.rho_d / 3600) # distillate mass flow rate (kg/s)
m.m_b_cal = Constraint(expr = m.m_b == m.q_b * m.rho_b / 3600) # brine mass flow rate (kg/s)
m.m_f_cal = Constraint(expr = m.m_f == m.qF * m.rho_f / 3600) # feed mass flow rate (kg/s)
#m.m_sw_cal= Constraint(expr = m.m_sw * (m.h_cool-m.h_sw) == ((1-m.Q_loss)*m.P_req - m.m_b * m.h_b - m.m_d * m.h_d + m.h_cool * m.m_f)  )
m.m_sw_cal= Constraint(expr = m.m_sw== ((1-m.Q_loss)*m.P_req - m.m_b * m.h_b - m.m_d * m.h_d + m.h_cool * m.m_f) / (m.h_cool - m.h_sw)) # intake water mass flow rate (kg/s)

# Volume flow rates
m.q_sw_cal = Constraint(expr = m.q_sw == m.m_sw / m.rho_sw * 3600) # m3/h
m.q_cooling_cal = Constraint(expr = m.q_cooling == m.q_sw - m.qF)

In [4]:
'''
Cost Model Variables
'''

m.cost_sys = Var(initialize = 1500,
                 bounds = (0, None),
                 doc = 'System capital cost ($)')

m.HEX_area = Var(initialize = 300,
              bounds = (0, None),
              units = pyunits.m**2 / (pyunits.kg / pyunits.s),
              doc = 'Intake water mass flow rate (kg/s)')

m.Prod = Var(initialize = 328500,
             bounds = (0, None),
             units = pyunits.m**3,
             doc = 'Annual water production (m3)')

m.storage_cap = Var(initialize = 1,
                    bounds = (0, None),
                    units = pyunits.kWh,
                    doc = 'Thermal storage capacity (kWh)')

m.CAPEX = Var(initialize = 1,
              bounds = (0, None),
              doc = 'Unit capital cost ($/m3)')

m.OPEX =  Var(initialize = 1,
              bounds = (0, None),
              doc = 'Unit operational cost ($/m3)')

m.LCOW =  Var(initialize = 1,
              bounds = (0, None),
              doc = 'Levelized cost of water ($/m3)')

'''
Cost Model Parameters (Inputs)
'''

m.downtime = Param(initialize = 10, 
                   doc = "Annual downtime (%)")

m.f_HEX = Param(initialize = 0.4,
                   doc = "Cost fraction of the evaporator")

m.SEEC = Param(initialize = 1.5,
               units = pyunits.kWh / pyunits.m**3,
               doc = "Specific electricity energy consumption (kWh/m3)")

m.coe = Param(initialize = 0.04,
              doc = "Unit cost of electricity ($/kWh)")

m.coh = Param(initialize = 0.01,
              doc ="Unit cost of heat ($/kWh)")

m.Chemicals = Param(initialize = 0.04,
                    doc = "Unit chemical cost ($/m3)")

m.Labor = Param(initialize = 0.033,
                doc = "Unit labor cost ($/m3)")

m.Discharge = Param(initialize = 0.02,
                      doc = "Unit water disposal cost ($/m3)")

m.Maintenance = Param(initialize = 2,
                      doc = "Maintenance cost (percentage of the capital cost)")

m.Insurance = Param(initialize = 0.5,
                    doc = "Insurance (percentage of the capital cost)")

m.Miscellaneous = Param(initialize = 0.1,
                        doc = "Unit miscellaneous cost ($/m3)")

m.yrs = Param(initialize = 20,
              doc = "Design plant lifetime (yrs)")

m.int_rate = Param(initialize = 0.04,
                   doc = "Average interest rate")

m.cost_storage = Param(initialize = 26,
                       doc = "Cost of thermal storage ($/kWh)")

m.storage_hr = Param(initialize = 0,
                     doc = "Storage hour (hrs)")


'''
Add equations for cost models
'''
# Calculate heat exchanger area in m2/(kg/s)
m.HEX_area_cal = Constraint(expr = m.HEX_area == m.sA * 86.4) # 86.4 is the conversion factor from m2/(m3/day) to m2/(kg/s)

# Calculate storage capacity (kWh)
m.storage_cap_cal = Constraint(expr = m.storage_cap == m.storage_hr * m.P_req)

# Calculate annual water production (m3)
m.Prod_cal = Constraint(expr = m.Prod == m.Capacity * 365 * (1 - m.downtime/100) )

# MED system cost ($)
m.cost_sys_cal = Constraint(expr = m.cost_sys == 6291 * m.Capacity**(-0.135) * ( 1- m.f_HEX + m.f_HEX * (m.HEX_area / 302.01)**0.8) )


# Unit Capital cost ($/m3)
m.CAPEX_cal = Constraint(expr = m.CAPEX == (m.cost_sys * m.Capacity + m.cost_storage * m.storage_cap) * m.int_rate * (1+m.int_rate)** m.yrs \
                / ((1+m.int_rate)**m.yrs -1) / m.Prod )

                         
# Unit OPEX ($/m3)
m.OPEX_cal = Constraint(expr = m.OPEX == m.STEC * m.coh + m.SEEC * m.coe + m.Chemicals + m.Labor + m.Maintenance/100 * m.cost_sys * m.Capacity / m.Prod \
                             + m.Miscellaneous + m.Discharge + m.Insurance/100 * m.cost_sys * m.Capacity / m.Prod )                         

# LCOW ($/m3)
m.LCOW_cal = Constraint(expr = m.LCOW == m.CAPEX + m.OPEX)



In [5]:
'''
Sample simulation
'''

# Input initialization

# Nef is declared in the beginning and it should also be an input to the model
m.Xf.fix(35000)
m.Ts.fix(80)
m.Capacity.fix(2000)
m.Tin.fix(15)
m.RR.fix(0.5)

print(f'Degrees of Freedom: {degrees_of_freedom(m)}')
solver = SolverFactory('ipopt')

results = solver.solve(m)

print(results)
m.display()

Degrees of Freedom: 0

Problem: 
- Lower bound: -inf
  Upper bound: inf
  Number of objectives: 1
  Number of constraints: 35
  Number of variables: 35
  Sense: unknown
Solver: 
- Status: ok
  Message: Ipopt 3.13.2\x3a Optimal Solution Found
  Termination condition: optimal
  Id: 0
  Error rc: 0
  Time: 0.047833919525146484
Solution: 
- number of solutions: 0
  number of solutions displayed: 0

Model unknown

  Variables:
    Xf : Feedwater salinity (g/L)
        Size=1, Index=None, Units=mg/l
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None : 30000 : 35000 : 60000 :  True :  True :  Reals
    Ts : The temperature of the steam at the inlet of the first bundle tube (C)
        Size=1, Index=None, Units=C
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :    60 :    80 :    85 :  True :  True :  Reals
    Capacity : Capacity of the plant (m3/day)
        Size=1, Index=None, Units=m**3/d
        Key  : Lower : Value : Upper : Fixed : Sta

In [6]:
'''
Test case 1 for optimization
'''

# Unfix Tin and RR
m.Tin.unfix()
m.RR.unfix()

# Set the objective to minimize LCOW
m.obj1 = Objective(rule=m.LCOW)

opt_results = solver.solve(m)

print('Optimized inputs: ')
print(m.Tin.display())
print(m.RR.display())

print('Optimized outputs: ')
print(m.GOR.display())
print(m.sA.display())
print(m.STEC.display())
print(m.LCOW.display())
# We see that Ts = 85 is better for than Ts = 80. 

Optimized inputs: 
Tin : Condenser inlet seawater temperature (C)
    Size=1, Index=None, Units=C
    Key  : Lower : Value              : Upper : Fixed : Stale : Domain
    None :    15 : 20.438704109228432 :    35 : False : False :  Reals
None
RR : Recovery ratio
    Size=1, Index=None
    Key  : Lower : Value              : Upper : Fixed : Stale : Domain
    None :   0.3 : 0.4999999607146561 :   0.5 : False : False :  Reals
None
Optimized outputs: 
GOR : Gained output ratio
    Size=1, Index=None
    Key  : Lower : Value             : Upper : Fixed : Stale : Domain
    None :     0 : 9.849273997720582 :  None : False : False :  Reals
None
sA : Specific area (m2/m3/day))
    Size=1, Index=None, Units=d/m
    Key  : Lower : Value              : Upper : Fixed : Stale : Domain
    None :     0 : 3.7556147842819554 :  None : False : False :  Reals
None
STEC : Specific thermal power consumption (kWh/m3)
    Size=1, Index=None, Units=kWh/m**3
    Key  : Lower : Value             : Upper : F

In [7]:
'''
Test case 2 for optimization
'''

# Unfix Ts
m.Ts.unfix()

# Fix the unfixed variables in Test case 1
m.Tin.fix(15)
m.RR.fix(0.5)

# Set the objective to minimize LCOW
m.obj1 = Objective(rule=m.LCOW)

opt_results = solver.solve(m)

print('Optimized inputs: ')
print(m.Ts.display())

print('Optimized outputs: ')
print(m.GOR.display())
print(m.sA.display())
print(m.STEC.display())
print(m.LCOW.display())
# We see that Ts = 85 is better for than Ts = 80. 

    'pyomo.core.base.objective.ScalarObjective'>) on block unknown with a new
    Component (type=<class 'pyomo.core.base.objective.ScalarObjective'>). This
    is usually indicative of a modelling error. To avoid this warning, use
    block.del_component() and block.add_component().
Optimized inputs: 
Ts : The temperature of the steam at the inlet of the first bundle tube (C)
    Size=1, Index=None, Units=C
    Key  : Lower : Value : Upper : Fixed : Stale : Domain
    None :    60 :  85.0 :    85 : False : False :  Reals
None
Optimized outputs: 
GOR : Gained output ratio
    Size=1, Index=None
    Key  : Lower : Value             : Upper : Fixed : Stale : Domain
    None :     0 : 9.690619201828827 :  None : False : False :  Reals
None
sA : Specific area (m2/m3/day))
    Size=1, Index=None, Units=d/m
    Key  : Lower : Value              : Upper : Fixed : Stale : Domain
    None :     0 : 3.0242750983411493 :  None : False : False :  Reals
None
STEC : Specific thermal power consumptio

In [5]:
# The results are identical to the previous Jupyter Notebook model. 
# Then we test if the solution can be found with different inputs.
'''
Test more input combinations
'''
import itertools

# Pick 3 values within the limit of each input variables 
# Note: Nef is tested manually since it's a parameter and the results shows no error as well.
    
Xfs = [30000,45000,60000]
RRs = [0.3,0.4,0.5]
Tins = [15,25,35]
Tss = [60,70,85]
Caps = [2000,10000,100000]

# Test the model for each of the input combination
test_results = {}

for (Xf, RR, Tin, Ts, Cap) in itertools.product(Xfs, RRs, Tins, Tss, Caps):
    m.Xf.fix(Xf)
    m.Ts.fix(Ts)
    m.Capacity.fix(Cap)
    m.Tin.fix(Tin)
    m.RR.fix(RR)    
    solver = SolverFactory('ipopt')
    try:
        results = solver.solve(m)
        test_results[(Xf, RR, Tin, Ts, Cap)] = (m.GOR(), m.sA())
    except:
        print('Model failed with the inputs of (Xf, RR, Tin, Ts, Cap) = ', Xf, RR, Tin, Ts, Cap)

In [6]:
# No error generated

# We can print the empirical results and compare with the EES model results if needed.
for key, val in test_results.items():
    print( key, ': ', val)

(30000, 0.3, 15, 60, 2000) :  (9.407554951260002, 4.831827809999956)
(30000, 0.3, 15, 60, 10000) :  (9.40755495126004, 4.831827809989083)
(30000, 0.3, 15, 60, 100000) :  (9.40755495126004, 4.831827809989082)
(30000, 0.3, 15, 70, 2000) :  (9.26726835726004, 4.102990504993935)
(30000, 0.3, 15, 70, 10000) :  (9.26726835726004, 4.102990504993937)
(30000, 0.3, 15, 70, 100000) :  (9.26726835726004, 4.102990504993935)
(30000, 0.3, 15, 85, 2000) :  (9.045250966260044, 1.7471864412511797)
(30000, 0.3, 15, 85, 10000) :  (9.045250966260042, 1.7471864412511797)
(30000, 0.3, 15, 85, 100000) :  (9.045250966260042, 1.7471864412511797)
(30000, 0.3, 25, 60, 2000) :  (9.537025321260002, 6.874134599999943)
(30000, 0.3, 25, 60, 10000) :  (9.537025321260002, 6.874134599999946)
(30000, 0.3, 25, 60, 100000) :  (9.537025321260002, 6.874134599999943)
(30000, 0.3, 25, 70, 2000) :  (9.405018727260002, 4.697276534999967)
(30000, 0.3, 25, 70, 10000) :  (9.40501872726, 4.697276534999967)
(30000, 0.3, 25, 70, 100000